In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('/content/revenue 20-2025.csv')

# Cek kolom
print(df.columns)

In [ ]:
# Ubah kolom tanggal (sesuaikan jika berbeda)
df['Tanggal'] = pd.to_datetime(df['order_date'])

# Pastikan revenue numeric
df['Revenue'] = pd.to_numeric(df['revenue'], errors='coerce')

# Drop null
df = df.dropna(subset=['Tanggal', 'Revenue'])


In [ ]:
df['Year'] = df['Tanggal'].dt.year
df['Month'] = df['Tanggal'].dt.month
df['Month_Name'] = df['Tanggal'].dt.strftime('%b')
df['Quarter'] = df['Tanggal'].dt.to_period('Q')
df['Q'] = df['Tanggal'].dt.quarter


In [ ]:
monthly_revenue = df.groupby(['Year','Month','Month_Name'])['Revenue'].sum().reset_index()

monthly_revenue = monthly_revenue.sort_values(['Year','Month'])

monthly_revenue.sample(5)


In [ ]:
threshold = monthly_revenue['Revenue'].quantile(0.75)

monthly_revenue['Kategori'] = np.where(
    monthly_revenue['Revenue'] >= threshold,
    'High Revenue',
    'Normal'
)

monthly_revenue.sort_values('Revenue', ascending=False).sample(10)


In [ ]:
plt.figure(figsize=(14,6))
sns.barplot(data=monthly_revenue, x='Month_Name', y='Revenue', hue='Kategori')
plt.title('Monthly Revenue Classification (2020-2025)')
plt.xticks(rotation=45)
plt.savefig('monthly_revenue_classification.png')
plt.show()


In [ ]:
quarterly_revenue = df.groupby(['Year','Q'])['Revenue'].sum().reset_index()

quarterly_revenue = quarterly_revenue.sort_values(['Year','Q'])

quarterly_revenue.head()


In [ ]:
q_threshold = quarterly_revenue['Revenue'].quantile(0.75)

quarterly_revenue['Kategori'] = np.where(
    quarterly_revenue['Revenue'] >= q_threshold,
    'High Revenue',
    'Normal'
)

quarterly_revenue.sort_values('Revenue', ascending=False)
quarterly_revenue.to_csv('quarterly_revenue_classification.csv', index=False)


In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=quarterly_revenue, x='Q', y='Revenue', hue='Year', palette='viridis')
plt.xlabel('Quarter')
plt.ylabel('Revenue')
plt.title('Quarterly Revenue per Year')
plt.savefig('quarterly_revenue_classification.png')
plt.show()


In [ ]:
pivot = monthly_revenue.pivot_table(
    values='Revenue',
    index='Month',
    columns='Year'
)

pivot.to_csv('monthly_revenue_pivot.csv')
